# 01 - Data Acquisition & Preprocessing

This notebook fetches satellite (Sentinel-2), weather (ERA5), and soil (SoilGrids) data for the region of interest using Google Earth Engine. If GEE is not available, it generates synthetic data for offline development.

## 1. Setup and Authentication

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
from pathlib import Path

# ROI: Kansas, USA (small agricultural area)
ROI_COORDS = [[-97.5, 38.5], [-97.0, 38.5], [-97.0, 39.0], [-97.5, 39.0]]
START_DATE = '2023-04-01'
END_DATE = '2023-07-31'
TARGET_YEAR = 2023

In [ ]:
# Google Earth Engine authentication (run: earthengine authenticate)
try:
    import ee
    ee.Initialize()
    GEE_AVAILABLE = True
    print('Earth Engine initialized successfully.')
except Exception as e:
    GEE_AVAILABLE = False
    print(f'GEE not available: {e}. Using synthetic data.')

## 2. Sentinel-2 with Cloud Masking and 10-Day Composites

In [ ]:
if GEE_AVAILABLE:
    # Load Sentinel-2 L2A, apply QA60 cloud mask, create composites
    roi = ee.Geometry.Polygon(ROI_COORDS + [ROI_COORDS[0]])
    collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(roi)
        .filterDate(START_DATE, END_DATE)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)))
    
    def mask_clouds(img):
        qa = img.select('QA60')
        cloud_bit = 1 << 10
        cirrus_bit = 1 << 11
        mask = qa.bitwiseAnd(cloud_bit).eq(0).And(qa.bitwiseAnd(cirrus_bit).eq(0))
        return img.updateMask(mask)
    
    def add_indices(img):
        ndvi = img.normalizedDifference(['B8', 'B4']).rename('NDVI')
        evi = img.expression(
            '2.5 * (NIR - RED) / (NIR + 6*RED - 7.5*BLUE + 1)',
            {'NIR': img.select('B8'), 'RED': img.select('B4'), 'BLUE': img.select('B2')}).rename('EVI')
        savi = img.expression(
            '1.5 * (NIR - RED) / (NIR + RED + 0.5)',
            {'NIR': img.select('B8'), 'RED': img.select('B4')}).rename('SAVI')
        return img.addBands([ndvi, evi, savi])
    
    masked = collection.map(mask_clouds).map(add_indices)
    composite = masked.median()
    print('Sentinel-2 composite created.')
else:
    print('Skipping GEE - using synthetic data.')

## 3. ERA5 Weather and SoilGrids

In [ ]:
if GEE_AVAILABLE:
    era5 = ee.ImageCollection('ECMWF/ERA5/DAILY').filterBounds(roi).filterDate(START_DATE, END_DATE)
    print('ERA5 loaded.')
    # SoilGrids: use OPENLANDMAP or similar if available in GEE
    print('Soil data: use SoilGrids from ISRIC or GEE catalog.')

## 4. Extract Values per Field & Save

In [ ]:
from src.synthetic_data import generate_synthetic_dataset
from src.utils import get_data_dir

# Use synthetic data (realistic relationships) when GEE extraction is complex
# In production: use ee.Image.sampleRegions() with field polygons
df = generate_synthetic_dataset(
    n_fields=100,
    start_date=START_DATE,
    end_date=END_DATE,
    target_year=TARGET_YEAR,
    seed=42
)

data_dir = get_data_dir()
data_dir.mkdir(parents=True, exist_ok=True)
df.to_csv(data_dir / 'merged_data.csv', index=False)
print(f'Saved {len(df)} rows to {data_dir / "merged_data.csv"}')
df.head()